Import Necessary Dependencies

In [2]:
import torch
import time
import numpy as np

**Problem 1** (Practical 1)

In [ ]:
x=torch.full((13, 13), 1, dtype=torch.int32)
x[:, [1, 6, 11]]=2
x[[1, 6, 11], :]=2
x[3:5, 3:5]=3
x[8:10, 3:5]=3
x[3:5, 8:10]=3
x[8:10, 8:10]=3
x

**Problem 2** (Practical)

In [ ]:
x=torch.empty((20, 20), dtype=torch.float32)
x.normal_(mean=0.0, std=1.0)
D=torch.diag(torch.arange(1, 21).float())
new_matrix=torch.inverse(x) @ D @ x
eigenvalues=torch.linalg.eigvals(new_matrix).real.sort().values
eigenvalues

**Problem 3** (Practical)

In [ ]:
x_1=torch.normal(mean=0.0, std=1.0, size=(5000, 5000), dtype=torch.float16, device='cuda')
x_2=torch.normal(mean=0.0, std=1.0, size=(5000, 5000), dtype=torch.float16, device='cuda')

torch.cuda.synchronize()
start_time=time.perf_counter()
result=x_1.mm(x_2)
torch.cuda.synchronize() 
end_time=time.perf_counter()
elapsed_time=end_time - start_time
print(f"Time taken for matrix multiplication: {elapsed_time:.4f} seconds")
print(f"Flops per Second: {(5000 * 5000 * (5000 + 4999)) / elapsed_time:.2f}")

**Problem 4** (Practical)

In [ ]:
def mul_row(A: torch.Tensor) -> torch.Tensor:
    mult_tensor=torch.arange(1, A.shape[0] + 1, 1, device=A.device, dtype=A.dtype)[:, None]
    return A * mult_tensor

def mul_row_faster(A: torch.Tensor) -> torch.Tensor:
    mult_tensor=torch.arange(1, A.shape[0] + 1, 1, device=A.device, dtype=A.dtype).unsqueeze(1)
    A*=mult_tensor
    return A

default=torch.ones((10000, 4000), device='cuda', dtype=torch.int64)
torch.cuda.synchronize() 
start_time=time.perf_counter()
mul_row(default)
torch.cuda.synchronize() 
end_time=time.perf_counter()
elapsed_time=end_time - start_time
print(f"Time taken for mul_row: {elapsed_time:.4f} seconds")

torch.cuda.synchronize() 
start_time=time.perf_counter()
mul_row_faster(default)
torch.cuda.synchronize() 
end_time=time.perf_counter()
elapsed_time=end_time - start_time
print(f"Time taken for mul_row_faster: {elapsed_time:.4f} seconds")

**Problem 1** (Homework)

Firstly, a redundancy is introduced in the first line when you look to set the values to be the floating point value '1.0', but then set the data type (dtype) to be an 8-bit integer. 

The next row has the tensor replacing the indexes (3, 3) and (5, 5) with threes, which is far from ideal. Furthermore, since the tensor assumes you are trying to modify the value indexed at (5, 5) as opposed to the intended (4, 4), a three ends up in the wrong place.

Finally, the second and third to last lines include double colons. Even if they didn't have double colons, the columns and rows ranging from 1(included) to 5(excluded) would simply be overriden with twos, erasing certain progress made prior.

**Problem 2** (Homework)

a) For a diagonal matrix, the diagonal entries are the eigenvalues of that matrix.

So the eigenvalues of $D$ are the diagonal entries of $D$.

Furthermore, if you let a matrix $A$ be represented as the following eigendecomposition $A=M^{-1}DM$, then $D$ will inherently be the eigenvalues of $A$.

Therefore the eigenvalues of $D$ is equal to the eigenvalues of $A=M^{-1}DM$

b) The torch.linalg.eigh(tensor) is designed for symmetric matrices, so it removes half of the information (represents it as one triangular matrix).

**Problem 3** (Homework)

In [ ]:
x_1=torch.normal(mean=0.0, std=1.0, size=(5000, 5000), dtype=torch.float16, device='cuda')
x_2=torch.normal(mean=0.0, std=1.0, size=(5000, 5000), dtype=torch.float16, device='cuda')

times=np.array([])
for _ in range(10):
    torch.cuda.synchronize()
    start_time=time.perf_counter()
    x_1.mm(x_2)
    torch.cuda.synchronize()
    end_time=time.perf_counter()
    times=np.append(times, end_time - start_time)

print(f"Average time taken for matrix multiplication: {times.mean():.4f} seconds")
print(f"Standard deviation of time taken for matrix multiplication: {times.std():.4f} seconds")
print(f"Flops per Second: {(5000 * 5000 * (5000 + 4999)) / times.mean():.2f}")



**Problem 4** (Homework)

In [ ]:
def mat_add_vals_slow(A):
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            A[i, j] += i + j
    return A

def mat_add_vals_fast(A):
    i_indices=torch.arange(A.shape[0], device=A.device, dtype=torch.int32).unsqueeze(1)
    j_indices=torch.arange(A.shape[1], device=A.device, dtype=torch.int32).unsqueeze(0)
    A += i_indices + j_indices
    return A

x=torch.zeros((1000, 400), device='cuda', dtype=torch.int8)
torch.cuda.synchronize()
start_time=time.perf_counter()
mat_add_vals_slow(x)
torch.cuda.synchronize()
end_time=time.perf_counter()
elapsed_time=end_time - start_time
print(f"Time taken for mat_add_vals_slow: {elapsed_time:.4f} seconds")

torch.cuda.empty_cache()

x=torch.zeros((10000, 10000), device='cuda', dtype=torch.float16)
torch.cuda.synchronize()
for _ in range(10):
    mat_add_vals_fast(x)
start_time=time.perf_counter()
mat_add_vals_fast(x)
torch.cuda.synchronize()
end_time=time.perf_counter()
elapsed_time=end_time - start_time
print(f"Time taken for mat_add_vals_fast: {elapsed_time:.10f} seconds")

# 1. Define the math
m, n = x.shape
total_ops = 2 * m * n  # 2 ops per element

# 2. Calculate FLOPS (or IOPS for int8)
# We use elapsed_time from your existing timer
flops = total_ops / elapsed_time
gflops = flops / 1e9  # Convert to GigaFLOPS for readability

print(f"Total Operations: {total_ops}")
print(f"Throughput: {gflops:.6f} GFLOPS")

In [ ]:
torch.cuda.empty_cache()
size = 8192
a = torch.randn((size, size), device='cuda', dtype=torch.float16)
b = torch.randn((size, size), device='cuda', dtype=torch.float16)

for _ in range(10):
    torch.matmul(a, b)

torch.cuda.synchronize()
start = time.perf_counter()
torch.matmul(a, b)
torch.cuda.synchronize()
end = time.perf_counter()

total_ops = 2 * (size**3)
tflops = (total_ops / (end - start)) / 1e12

print(f"Time: {end - start:.4f}s")
print(f"Throughput: {tflops:.2f} TFLOPS")

Time: 0.0162s
Throughput: 67.67 TFLOPS
